# OmniGDA-Manip — Offline GPU Subspace Distillation & Adapter Training (Colab)

OmniGDA-Manip performs **training-free** online few-shot & zero-shot registration at deployment time via tied-covariance Shrinkage GDA/GMDA.  
This notebook optionally runs on a **Google Colab GPU (T4 / L4 / A100)** to distill two lightweight linear projection adapters (`.npz`, zero-pickle) that improve discriminative geometry and dramatically cut covariance inversion latency on the RTX 3060 deployment box:

1. **3D Fisher-SupCon Subspace Adapter** (`colab/distill_3d_adapter.py`):
   - Compresses raw 108-D `Geometric3DEncoder` descriptors $\to$ **64-D** (`W: (64, 108), b: (64,)`)
   - Objective: $\mathcal{L} = \mathcal{L}_{\text{SupCon}} + \lambda_f \frac{\operatorname{tr}(S_W)}{\operatorname{tr}(S_B)+\epsilon} + \lambda_o \|W W^\top - I\|_F^2$
   - **2.85x** smaller 3D covariance matrix (`108x108 -> 64x64`), faster Cholesky precision inversion, and direct drop-in loading via `Geometric3DEncoder.load_projection_adapter(path)`.

2. **2D DINOv2 Cross-Model Relational Subspace Projector** (`colab/distill_2d_subspace.py`):
   - Distills pairwise relational similarity geometry (Gram MSE + Linear CKA + SupCon) from a larger DINOv2 teacher (`vit_base_patch14_dinov2.lvd142m` or `vit_large_patch14_dinov2.lvd142m`) into a compact **128-D** linear subspace on top of student `vit_small_patch14_dinov2.lvd142m` (`384 -> 128`).
   - **9.0x** smaller `SegmentGDAPlus` covariance matrix (`384x384 -> 128x128`).

## 1. Environment Check & Dependency Installation via `uv`

Checks Colab GPU availability, clones/enters the `omnigda-manip` repository, and installs required packages quickly with `uv`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

# 1. Inspect Colab GPU
!nvidia-smi

# 2. Install uv for fast deterministic package installation
!pip install -q uv

# 3. Locate or clone the repository (adjust REPO_URL if running from a remote fork)
REPO_DIR = Path("/content/omnigda-manip")
if not REPO_DIR.exists():
    if Path("src/omnigda").exists():
        REPO_DIR = Path.cwd()
    else:
        REPO_URL = os.environ.get("OMNIGDA_REPO_URL", "https://github.com/your-org/omnigda-manip.git")
        print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Install Python dependencies via uv into Colab's system environment
!uv pip install --system --quiet \
    numpy scipy scikit-learn pydantic pyyaml pillow opencv-python-headless \
    matplotlib timm open-clip-torch open3d mani-skill tqdm

print(f"Working directory: {Path.cwd()}")
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Training Hyperparameters & Artifact Paths

Configure output dimensions, DINOv2 teacher model (`vit_base` for T4/L4 or `vit_large` for L4/A100), epochs, and artifact paths.

In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("artifacts/colab_adapters")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 3D Fisher-SupCon Adapter settings
OUT_DIM_3D = 64
EPOCHS_3D = 150
NUM_TEMPLATE_SHOTS_3D = 8
NUM_CLUTTER_EPISODES_3D = 2
ADAPTER_3D_PATH = ARTIFACT_DIR / "geom3d_adapter_64d.npz"

# 2D DINOv2 Cross-Model Subspace Distillation settings
STUDENT_MODEL_2D = "vit_small_patch14_dinov2.lvd142m"
# Switch to "vit_large_patch14_dinov2.lvd142m" on Colab L4 / A100 for strongest teacher manifold
TEACHER_MODEL_2D = "vit_base_patch14_dinov2.lvd142m"
OUT_DIM_2D = 128
EPOCHS_2D = 150
NUM_SHOTS_2D = 6
TOKENS_PER_IMAGE_2D = 32
ADAPTER_2D_PATH = ARTIFACT_DIR / "dinov2_subspace_128d.npz"

# Set USE_SYNTHETIC_DEMO = True only for an ultra-fast 5-second dry run without ManiSkill/timm downloads
USE_SYNTHETIC_DEMO = False
SEED = 42

print("3D output:", ADAPTER_3D_PATH)
print("2D output:", ADAPTER_2D_PATH)

## 3. Train 3D Fisher-SupCon Discriminative Subspace Adapter (`108D -> 64D`)

Collects multi-view partial 3D point clouds from `capture_few_shot_templates` + `ClutteredTabletopSim`, extracts raw 108-D `Geometric3DEncoder` descriptors, trains the linear projection adapter on GPU, and evaluates held-out 3D GDA Top-1 accuracy & Cholesky inversion speedup.

In [ ]:
import numpy as np

from colab.distill_3d_adapter import (
    collect_multiview_3d_features,
    generate_synthetic_3d_features,
    train_3d_metric_adapter,
)

if USE_SYNTHETIC_DEMO:
    x_3d, y_3d, classes_3d = generate_synthetic_3d_features(
        num_classes=7, samples_per_class=28, in_dim=108, seed=SEED
    )
else:
    x_3d, y_3d, classes_3d = collect_multiview_3d_features(
        num_template_shots=NUM_TEMPLATE_SHOTS_3D,
        num_clutter_episodes=NUM_CLUTTER_EPISODES_3D,
        augmentations_per_view=1,
        seed=SEED,
    )

print(f"Collected 3D dataset: X={x_3d.shape}, classes={classes_3d}")

res_3d = train_3d_metric_adapter(
    features=x_3d,
    labels=y_3d,
    class_names=classes_3d,
    out_dim=OUT_DIM_3D,
    epochs=EPOCHS_3D,
    lr=2.5e-3,
    temperature=0.15,
    lambda_fisher=0.25,
    lambda_ortho=0.02,
    device="cuda",
    seed=SEED,
    output_path=ADAPTER_3D_PATH,
)

m3 = res_3d.metrics
print(f"3D Loss:           {m3['initial_loss']:.4f} -> {m3['final_loss']:.4f} ({m3['train_time_sec']:.2f}s)")
print(f"3D GDA Top-1 Acc:  {m3['raw_top1_acc']*100:.2f}% (108D) -> {m3['proj_top1_acc']*100:.2f}% (64D)")
print(f"3D Fisher Ratio:   {m3['raw_fisher_ratio']:.3f} -> {m3['proj_fisher_ratio']:.3f}")
print(f"3D Cov Reduction:  {m3['cov_reduction_factor']:.2f}x ({int(m3['raw_cov_elements'])} -> {int(m3['proj_cov_elements'])} elems)")
print(f"3D Cholesky Inv:   {m3['raw_cholesky_ms']:.3f} ms -> {m3['proj_cholesky_ms']:.3f} ms ({m3['cholesky_speedup']:.2f}x speedup)")

## 4. Train 2D DINOv2 Cross-Model Relational Subspace Projector (`384D -> 128D`)

Extracts spatially aligned foreground patch tokens from student `vit_small_patch14_dinov2.lvd142m` and teacher `vit_base`/`vit_large`, then optimizes Relational Cosine Distillation (Gram MSE + Linear CKA) + SupCon + Orthogonality to cut `SegmentGDAPlus` covariance storage by **9.0x**.

In [ ]:
from colab.distill_2d_subspace import (
    extract_dinov2_patch_dataset,
    generate_synthetic_2d_distill_data,
    train_2d_subspace_projector,
)

if USE_SYNTHETIC_DEMO:
    xs_2d, xt_2d, y_2d, classes_2d = generate_synthetic_2d_distill_data(
        num_classes=8, tokens_per_class=80, student_dim=384, teacher_dim=768, seed=SEED
    )
else:
    xs_2d, xt_2d, y_2d, classes_2d = extract_dinov2_patch_dataset(
        num_shots=NUM_SHOTS_2D,
        tokens_per_image=TOKENS_PER_IMAGE_2D,
        student_model=STUDENT_MODEL_2D,
        teacher_model=TEACHER_MODEL_2D,
        device="cuda",
        fp16=True,
        seed=SEED,
    )

print(f"Extracted 2D tokens: student={xs_2d.shape}, teacher={xt_2d.shape}, classes={classes_2d}")

res_2d = train_2d_subspace_projector(
    student_features=xs_2d,
    teacher_features=xt_2d,
    labels=y_2d,
    class_names=classes_2d,
    out_dim=OUT_DIM_2D,
    epochs=EPOCHS_2D,
    lr=2e-3,
    gram_weight=4.0,
    cka_weight=2.0,
    lambda_supcon=0.12,
    lambda_ortho=0.01,
    device="cuda",
    seed=SEED,
    output_path=ADAPTER_2D_PATH,
    student_model=STUDENT_MODEL_2D,
    teacher_model=TEACHER_MODEL_2D,
)

m2 = res_2d.metrics
print(f"2D Loss:           {m2['initial_loss']:.4f} -> {m2['final_loss']:.4f} ({m2['train_time_sec']:.2f}s)")
print(f"2D Relational CKA: {m2['raw_cka']:.4f} -> {m2['proj_cka']:.4f} | Gram MSE: {m2['raw_gram_mse']:.5f} -> {m2['proj_gram_mse']:.5f}")
print(f"2D GDA Top-1 Acc:  {m2['raw_top1_acc']*100:.2f}% (384D) -> {m2['proj_top1_acc']*100:.2f}% (128D)")
print(f"2D Cov Reduction:  {m2['cov_reduction_factor']:.2f}x ({int(m2['raw_cov_elements'])} -> {int(m2['proj_cov_elements'])} elems)")
print(f"2D Cholesky Inv:   {m2['raw_cholesky_ms']:.3f} ms -> {m2['proj_cholesky_ms']:.3f} ms ({m2['cholesky_speedup']:.2f}x speedup)")

## 5. Visualization: Training Curves, Before/After t-SNE, & Accuracy / Latency Summary

Inspects monotonic loss convergence, compares raw vs distilled feature manifolds via 2D t-SNE, and plots covariance compression & accuracy summaries.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from omnigda.core.gda_math import l2_normalize

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. 3D Loss curves
ax = axes[0, 0]
ax.plot(res_3d.history["total_loss"], label="Total", lw=2.2)
ax.plot(res_3d.history["supcon_loss"], label="SupCon", ls="--")
ax.plot(res_3d.history["fisher_loss"], label="Fisher", ls="-.")
ax.set_title("3D Fisher-SupCon Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)
ax.legend()

# 2. 2D Loss curves
ax = axes[0, 1]
ax.plot(res_2d.history["total_loss"], label="Total", lw=2.2)
ax.plot(res_2d.history["gram_loss"], label="Gram MSE", ls="--")
ax.plot(res_2d.history["cka_loss"], label="1 - CKA", ls="-.")
ax.plot(res_2d.history["supcon_loss"], label="SupCon", ls=":")
ax.set_title("2D DINOv2 Relational Distillation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)
ax.legend()

# 3. Accuracy & Covariance Reduction Bar Chart
ax = axes[0, 2]
labels_bar = ["3D GDA Acc", "2D GDA Acc", "2D CKA"]
raw_vals = [m3["raw_top1_acc"] * 100, m2["raw_top1_acc"] * 100, m2["raw_cka"] * 100]
proj_vals = [m3["proj_top1_acc"] * 100, m2["proj_top1_acc"] * 100, m2["proj_cka"] * 100]
x_pos = np.arange(len(labels_bar))
width = 0.35
ax.bar(x_pos - width / 2, raw_vals, width, label="Raw (108D / 384D)", color="#7f8c8d")
ax.bar(x_pos + width / 2, proj_vals, width, label="Distilled (64D / 128D)", color="#2980b9")
ax.set_xticks(x_pos)
ax.set_xticklabels(labels_bar)
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 108)
ax.set_title("Before vs After Subspace Distillation")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()

# 4 & 5. Before vs After t-SNE of 3D Descriptors
z_3d_proj = l2_normalize(x_3d @ res_3d.weight.T + res_3d.bias[None, :], axis=-1)
perp = max(5, min(30, x_3d.shape[0] // 4))
tsne_raw = TSNE(n_components=2, perplexity=perp, random_state=SEED).fit_transform(x_3d)
tsne_proj = TSNE(n_components=2, perplexity=perp, random_state=SEED).fit_transform(z_3d_proj)

for ax, emb2d, title in [
    (axes[1, 0], tsne_raw, "3D Raw Features (108D) — t-SNE"),
    (axes[1, 1], tsne_proj, "3D Fisher-SupCon Adapted (64D) — t-SNE"),
]:
    scatter = ax.scatter(emb2d[:, 0], emb2d[:, 1], c=y_3d, cmap="tab10", s=26, alpha=0.85)
    ax.set_title(title)
    ax.grid(True, alpha=0.25)

# 6. Cholesky Inversion Latency Comparison
ax = axes[1, 2]
chol_labels = ["3D GDA\n(108->64)", "2D SegmentGDA+\n(384->128)"]
raw_ms = [m3["raw_cholesky_ms"], m2["raw_cholesky_ms"]]
proj_ms = [m3["proj_cholesky_ms"], m2["proj_cholesky_ms"]]
x_c = np.arange(len(chol_labels))
ax.bar(x_c - width / 2, raw_ms, width, label="Raw Cov Inv (ms)", color="#e67e22")
ax.bar(x_c + width / 2, proj_ms, width, label="Distilled Cov Inv (ms)", color="#27ae60")
ax.set_xticks(x_c)
ax.set_xticklabels(chol_labels)
ax.set_ylabel("Latency (ms)")
ax.set_title("Cholesky Precision Inversion Speedup")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()

plt.tight_layout()
plot_path = ARTIFACT_DIR / "distillation_summary.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved summary figure to {plot_path}")

## 6. Verify Zero-Pickle Loading & Download Artifacts for RTX 3060 Deployment

Confirms that `geom3d_adapter_64d.npz` loads directly into `Geometric3DEncoder.load_projection_adapter(...)` and `dinov2_subspace_128d.npz` loads via `load_2d_projector(...)`, then triggers Google Colab file download.

In [ ]:
import numpy as np

from colab.distill_2d_subspace import load_2d_projector
from omnigda.perception_3d.encoders import Geometric3DEncoder

# 1. Verify 3D adapter loads seamlessly into Geometric3DEncoder
enc = Geometric3DEncoder()
enc.load_projection_adapter(ADAPTER_3D_PATH)
assert enc.feature_dim == OUT_DIM_3D
dummy_cloud = np.random.default_rng(SEED).normal(0.0, 0.05, size=(128, 3)).astype(np.float32)
feat_64 = enc.encode(dummy_cloud)
assert feat_64.shape == (OUT_DIM_3D,)
assert np.isclose(np.linalg.norm(feat_64), 1.0, atol=1e-4)
print(f"[OK] Geometric3DEncoder loaded {ADAPTER_3D_PATH.name} -> output shape={feat_64.shape}, norm={np.linalg.norm(feat_64):.5f}")

# 2. Verify 2D subspace projector round-trip
w_2d, b_2d, meta_2d = load_2d_projector(ADAPTER_2D_PATH)
assert w_2d.shape == (OUT_DIM_2D, 384)
assert b_2d.shape == (OUT_DIM_2D,)
print(f"[OK] 2D Projector loaded {ADAPTER_2D_PATH.name} -> W={w_2d.shape}, b={b_2d.shape}, cov_reduction={meta_2d['metrics']['cov_reduction_factor']:.1f}x")

# 3. Trigger download when running inside Google Colab
try:
    from google.colab import files  # type: ignore[import-not-found]
    files.download(str(ADAPTER_3D_PATH))
    files.download(str(ADAPTER_2D_PATH))
except ImportError:
    print("Not running inside Google Colab; artifacts saved locally at:")
    print(f"  - {ADAPTER_3D_PATH.resolve()}")
    print(f"  - {ADAPTER_2D_PATH.resolve()}")